
# 21 — Automated Prompt Optimization with DSPy

**Learning reference:** [BrightPool — Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course)

**Adapted from:**
- `dspy.ipynb`
- `funeval.ipynb`
- `dspy_primer_with_every.ipynb`

## Objective

Demonstrate automated prompt optimization using
DSPy and real Gemini model predictions.

The experiment includes:

1. Defining a DSPy signature and module.
2. Creating labeled training and test datasets.
3. Evaluating an unoptimized program.
4. Applying the BootstrapFewShot optimizer.
5. Evaluating the optimized program on held-out data.
6. Comparing accuracy and inspecting optimization results.

**Task:** Route customer-support messages to the
correct category: billing, delivery, returns or technical.

The objective is to measure whether automatic
few-shot optimization improves classification accuracy.



## 1. DSPy Fundamentals

**Signature**

Defines what information the model receives
and what it should return.

Example: `message -> category`

**Module**

A reusable Python component that executes
the language-model task.

**Metric**

Evaluates predictions against reference labels.
This experiment uses exact-match accuracy.

**BootstrapFewShot**

A DSPy optimizer that builds a prompt using
labeled examples and model-generated demonstrations
that satisfy the evaluation metric.

**Compilation**

DSPy's term for producing an optimized version
of a program. It does not fine-tune model weights.

**Held-Out Testing**

Measures baseline and optimized performance
on examples excluded from the training set.

Unlike manual prompt refinement, DSPy can
automate the selection of prompt demonstrations.


In [1]:

import pandas as pd

TRAIN = [
    ("I was charged twice for my monthly plan. Please reverse the extra payment.", "billing"),
    ("The tracking page says delivered, but no parcel arrived. Please locate it.", "delivery"),
    ("The shoes are too small. How can I send them back?", "returns"),
    ("The app crashes whenever I attempt to sign in.", "technical"),
    ("My order arrived, but the discount was not applied to the invoice.", "billing"),
    ("I need an updated arrival estimate for my delayed shipment.", "delivery"),
    ("The headphones work, but I want to return them within the return window.", "returns"),
    ("My payment went through, but the website shows an unexpected login error.", "technical")
]

TEST = [
    ("The package is late, but my main concern is being charged twice. Please fix the charge.", "billing"),
    ("I received a refund confirmation, but the replacement parcel has not arrived. Where is it?", "delivery"),
    ("The jacket arrived yesterday, but it is the wrong size. Please arrange a return.", "returns"),
    ("My subscription is active, but the mobile app freezes when I open it.", "technical"),
    ("I can log in now, but the renewal price is incorrect. Please correct the invoice.", "billing"),
    ("The product is fine, but the courier has not delivered the second box.", "delivery"),
    ("The delivery was delayed, but I have the item now and want to return it.", "returns"),
    ("The order was delivered, but the password reset link keeps failing.", "technical")
]

LABELS = {"billing", "delivery", "returns", "technical"}

assert len(TRAIN) == 8
assert len(TEST) == 8
assert all(label in LABELS for _, label in TRAIN + TEST)

print("Training examples:", len(TRAIN))
print("Held-out test examples:", len(TEST))

display(pd.DataFrame(
    TEST, columns=["Message", "Expected category"]
))


Training examples: 8
Held-out test examples: 8


,Message,Expected category
0,"The package is late, but my main concern is be...",billing
1,"I received a refund confirmation, but the repl...",delivery
2,"The jacket arrived yesterday, but it is the wr...",returns
3,"My subscription is active, but the mobile app ...",technical
4,"I can log in now, but the renewal price is inc...",billing
5,"The product is fine, but the courier has not d...",delivery
6,"The delivery was delayed, but I have the item ...",returns
7,"The order was delivered, but the password rese...",technical


In [2]:

%pip install -q --upgrade dspy

import dspy
from getpass import getpass

MODEL = "gemini-3.5-flash-lite"

lm = dspy.LM(
    f"gemini/{MODEL}",
    api_key=getpass("Gemini API key: "),
    temperature=0,
    max_tokens=256
)

dspy.configure(lm=lm)

print("DSPy configured with:", MODEL)


Gemini API key: ··········
DSPy configured with: gemini-3.5-flash-lite


In [3]:

class RouteTicket(dspy.Signature):
    """Classify the customer's primary requested action.
    Return exactly one category: billing, delivery,
    returns or technical. Prioritize the current request
    over background context.
    """

    message: str = dspy.InputField()
    category: str = dspy.OutputField()


class TicketRouter(dspy.Module):
    def __init__(self):
        super().__init__()
        self.classifier = dspy.Predict(RouteTicket)

    def forward(self, message):
        return self.classifier(message=message)


def normalize_label(value):
    return str(value).strip().lower()


def accuracy_metric(example, prediction, trace=None):
    return normalize_label(prediction.category) == example.category


trainset = [
    dspy.Example(message=message, category=label)
    .with_inputs("message")
    for message, label in TRAIN
]

testset = [
    dspy.Example(message=message, category=label)
    .with_inputs("message")
    for message, label in TEST
]

baseline = TicketRouter()

print("DSPy program and evaluation metric prepared.")


DSPy program and evaluation metric prepared.


In [4]:

def evaluate_router(program, examples):
    rows = []

    for index, example in enumerate(examples, start=1):
        prediction = program(message=example.message)

        predicted = normalize_label(prediction.category)
        expected = example.category

        rows.append({
            "Case": index,
            "Expected": expected,
            "Predicted": predicted,
            "Correct": predicted == expected
        })

    return pd.DataFrame(rows)


if "baseline_results" not in globals():
    baseline_results = evaluate_router(
        baseline, testset
    )

baseline_accuracy = baseline_results["Correct"].mean()

print("BASELINE RESULTS")
display(baseline_results)

print(f"Baseline accuracy: {baseline_accuracy:.1%}")


BASELINE RESULTS


,Case,Expected,Predicted,Correct
0,1,billing,billing,True
1,2,delivery,delivery,True
2,3,returns,returns,True
3,4,technical,technical,True
4,5,billing,billing,True
5,6,delivery,delivery,True
6,7,returns,returns,True
7,8,technical,technical,True


Baseline accuracy: 100.0%



## 2. Automatic Few-Shot Optimization

The baseline uses the task signature without
training demonstrations.

BootstrapFewShot optimizes the program by:

1. Running the program on training examples.
2. Checking predictions using the defined metric.
3. Retaining successful demonstrations.
4. Combining these with labeled examples.
5. Creating a compiled program containing
   additional examples in its prompt.

The optimized program uses the same Gemini model.

Only the prompt context changes; model weights
are not trained.

The test examples remain excluded from optimization.


In [5]:

from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(
    metric=accuracy_metric,
    max_bootstrapped_demos=2,
    max_labeled_demos=2,
    max_rounds=1
)

optimized = optimizer.compile(
    TicketRouter(),
    trainset=trainset
)

print("Optimization completed.")

demos = optimized.classifier.demos

print("Demonstrations selected:", len(demos))

for index, demo in enumerate(demos, start=1):
    print(
        f"{index}. {demo.message[:85]}... "
        f"-> {demo.category}"
    )


 25%|██▌       | 2/8 [00:01<00:03,  1.93it/s]

Bootstrapped 2 full traces after 2 examples for up to 1 rounds, amounting to 2 attempts.
Optimization completed.
Demonstrations selected: 2
1. I was charged twice for my monthly plan. Please reverse the extra payment.... -> billing
2. The tracking page says delivered, but no parcel arrived. Please locate it.... -> delivery


In [8]:

import time
from dspy.clients.errors import LMRateLimitError

if "optimized_rows" not in globals():
    optimized_rows = {}

for index, example in enumerate(testset, start=1):
    if index in optimized_rows:
        continue

    for attempt in range(5):
        try:
            prediction = optimized(
                message=example.message
            )

            predicted = normalize_label(
                prediction.category
            )

            optimized_rows[index] = {
                "Case": index,
                "Expected": example.category,
                "Predicted": predicted,
                "Correct": predicted == example.category
            }

            print(f"Case {index} completed.")
            break

        except LMRateLimitError:
            if attempt == 4:
                raise

            print("Rate limit reached. Retrying...")
            time.sleep(35)

    time.sleep(5)

optimized_results = pd.DataFrame(
    [optimized_rows[i] for i in sorted(optimized_rows)]
)

optimized_accuracy = optimized_results["Correct"].mean()

print("\nOPTIMIZED RESULTS")
display(optimized_results)

print(f"Optimized accuracy: {optimized_accuracy:.1%}")



OPTIMIZED RESULTS


,Case,Expected,Predicted,Correct
0,1,billing,billing,True
1,2,delivery,delivery,True
2,3,returns,returns,True
3,4,technical,technical,True
4,5,billing,billing,True
5,6,delivery,delivery,True
6,7,returns,returns,True
7,8,technical,technical,True


Optimized accuracy: 100.0%


In [9]:

comparison = pd.DataFrame({
    "Case": baseline_results["Case"],
    "Expected": baseline_results["Expected"],
    "Baseline": baseline_results["Predicted"],
    "Optimized": optimized_results["Predicted"],
    "Baseline correct": baseline_results["Correct"],
    "Optimized correct": optimized_results["Correct"]
})

display(comparison)

summary = pd.DataFrame({
    "Program": ["Baseline", "DSPy optimized"],
    "Accuracy": [baseline_accuracy, optimized_accuracy],
    "Correct predictions": [
        int(baseline_results["Correct"].sum()),
        int(optimized_results["Correct"].sum())
    ]
})

print("HELD-OUT ACCURACY")
display(summary)

difference = optimized_accuracy - baseline_accuracy

print(f"Accuracy change: {difference:+.1%}")

if difference > 0:
    print("The optimized program performed better.")
elif difference < 0:
    print("The baseline performed better.")
else:
    print("Both programs achieved the same accuracy.")


,Case,Expected,Baseline,Optimized,Baseline correct,Optimized correct
0,1,billing,billing,billing,True,True
1,2,delivery,delivery,delivery,True,True
2,3,returns,returns,returns,True,True
3,4,technical,technical,technical,True,True
4,5,billing,billing,billing,True,True
5,6,delivery,delivery,delivery,True,True
6,7,returns,returns,returns,True,True
7,8,technical,technical,technical,True,True


HELD-OUT ACCURACY


,Program,Accuracy,Correct predictions
0,Baseline,1.0,8
1,DSPy optimized,1.0,8


Accuracy change: +0.0%
Both programs achieved the same accuracy.


In [10]:

optimized.save("dspy_optimized_router.json")

print("Optimized program saved.")
print("Selected demonstrations:", len(demos))

print("\nMISCLASSIFIED TEST CASES")

mistakes = comparison[
    ~comparison["Optimized correct"]
]

if mistakes.empty:
    print("No errors on the held-out test set.")
else:
    display(mistakes)


Optimized program saved.
Selected demonstrations: 2

MISCLASSIFIED TEST CASES
No errors on the held-out test set.



## Findings and Limitations

### What This Notebook Demonstrates

- Defining DSPy signatures and modules.
- Running real Gemini predictions through DSPy.
- Building labeled training and test datasets.
- Measuring exact-match classification accuracy.
- Applying automated few-shot optimization.
- Comparing baseline and compiled programs.
- Inspecting and saving selected demonstrations.

### Prompt Engineering Techniques

**Declarative Prompting**

DSPy signatures define inputs, outputs and task
requirements without manually formatting each prompt.

**Few-Shot Optimization**

BootstrapFewShot automatically selects demonstrations
using training examples and evaluation feedback.

**Metric-Driven Evaluation**

Exact-match accuracy determines whether
predicted categories match reference labels.

**Held-Out Testing**

Both programs are evaluated on the same examples
excluded from optimization.

### Relationship to the Udemy Examples

This notebook adapts signatures, modules,
BootstrapFewShot and evaluation techniques from
the course's DSPy examples.

More advanced optimizers, including MIPROv2,
are not implemented in this focused experiment.


### Observed Results

DSPy successfully compiled the customer-support
routing program using BootstrapFewShot.

The optimizer selected two demonstrations
from the training dataset:

- A billing-related support message.
- A delivery-related support message.

**Held-out evaluation**

| Metric | Baseline | DSPy optimized |
|---|---:|---:|
| Correct predictions | 8/8 | 8/8 |
| Accuracy | 100% | 100% |
| Misclassified cases | 0 | 0 |

The difference in held-out accuracy was
**0 percentage points**.

**Interpretation**

Both programs correctly classified all eight
held-out messages.

The baseline had already reached perfect accuracy,
leaving no room for measurable improvement on
this dataset.

DSPy optimization successfully added
few-shot demonstrations, but did not improve
the observed accuracy.

This illustrates a ceiling effect: a successful
optimization process does not necessarily produce
higher performance when the original program
already solves the evaluated task.

The results demonstrate automated prompt
optimization mechanics, not evidence that
the optimized program generalizes better.



### Limitations

- Small, fictional training and test datasets.
- Only one model and one classification task.
- Results are based on one evaluation run.
- Demonstrations may not generalize to other inputs.
- Exact-match accuracy does not measure confidence.
- Optimization may provide no measurable improvement
  when the baseline is already accurate.
- No statistical significance can be established.

### Conclusion

DSPy enables metric-driven optimization of
language-model programs.

BootstrapFewShot builds an optimized program
using examples selected through the training
and evaluation process.

Held-out testing determines whether those
changes improve accuracy on unseen inputs.

The experiment demonstrates automatic
prompt optimization without fine-tuning
the underlying language model.
